# 16 · Train/test split

**What this notebook does:** it splits the cleaned listings into a **training set** and a **test set**, following the
split policy in `docs/data_cleaning_decisions.md` (section 13), and saves which listing belongs to which side.

**What this notebook does NOT do:** no preprocessing (no imputation, encoding or scaling), no feature engineering and
no model. Those steps come later and are fitted on the **training set only**.

**The policy in short:**

| Rule | Why |
|---|---|
| Split by **group**, not by row: all listings with the same `property_group_id` go to the same side | copies of one property on both sides would let a model "remember" a price instead of predicting it |
| Use `GroupShuffleSplit` with a **fixed `random_state`** | the same split every time the notebook is run |
| **Never** split by row order or date | the rows are in scrape order, which follows price (Spearman 0.985), and posting date is tied to the price searches |
| Save the assignment `property_id` → train/test | every later experiment uses exactly the same test set |
| Check that price and property type look similar on both sides | a sanity check, not a rule |
| Use the test set **once**, for the final evaluation | anything learned from the test set makes the score optimistic |

**Input:** `data/processed/cleaned_listings.csv` (from notebook 15, not modified here).
**Output:** `data/processed/train_test_split.csv` with the columns `property_id`, `property_group_id`, `split`.

## 1. Setup and loading

We also take an MD5 fingerprint of the cleaned file. It is checked again at the end to prove this notebook did not
change it, and it is printed so that later notebooks can confirm they use the same data the split was made on.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
SPLIT_CSV = PROJECT_ROOT / "data" / "processed" / "train_test_split.csv"


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


cleaned_md5_at_start = file_md5(CLEANED_CSV)
print("MD5 of cleaned_listings.csv:", cleaned_md5_at_start)

MD5 of cleaned_listings.csv: 3a81a385a9aca28d5b70322e1e143c08


In [2]:
listings = pd.read_csv(CLEANED_CSV, low_memory=False)
print("Cleaned listings:", listings.shape)

# The things the split depends on must be in order before we start
assert len(listings) == 20300, "Unexpected row count: was notebook 15 re-run with different rules?"
assert listings["property_id"].is_unique, "property_id must identify each row exactly once"
assert listings["property_group_id"].notna().all(), "Every row needs a property_group_id"
assert listings["price"].notna().all(), "Every row needs a target"

print("property_id unique:                ", listings["property_id"].is_unique)
print("property_group_id missing values:  ", listings["property_group_id"].isna().sum())
print("Unique property_group_id:          ", listings["property_group_id"].nunique())

Cleaned listings: (20300, 76)
property_id unique:                 True
property_group_id missing values:   0
Unique property_group_id:           19475


## 2. Why a random row split is not enough

**What:** before doing the real split, we look at how many listings share a group, and what an ordinary random
**row** split (`train_test_split`) would do to those groups.

**Why:** 7.6% of listings are copies of another listing of the same property (reposts, several agents, Telecalling
accounts; notebook 06). A random row split ignores this, so some copies land in train and others in test. The model
could then score well on those test listings simply because it saw the same property, often with the same price,
during training.

This cell is only an illustration. Its result is **not** used.

**Evidence:** `docs/duplicate_handling.md`; spec section 13.

In [3]:
group_sizes = listings["property_group_id"].map(listings["property_group_id"].value_counts())
in_multi_listing_group = group_sizes > 1

print("Listings in a group of 2 or more:", in_multi_listing_group.sum(),
      f"({in_multi_listing_group.mean():.1%} of listings)")
print("Groups with 2 or more listings:  ", listings.loc[in_multi_listing_group, "property_group_id"].nunique())
print("Largest group:                   ", group_sizes.max(), "listings")

# Illustration only: an ordinary random ROW split
naive_train, naive_test = train_test_split(listings, test_size=0.2, random_state=42)
groups_in_both = set(naive_train["property_group_id"]) & set(naive_test["property_group_id"])
print()
print("A random ROW split would put", len(groups_in_both), "property groups on BOTH sides.")
print("That is the leakage the group-aware split prevents.")

Listings in a group of 2 or more: 1534 (7.6% of listings)
Groups with 2 or more listings:   709
Largest group:                    10 listings

A random ROW split would put 250 property groups on BOTH sides.
That is the leakage the group-aware split prevents.


## 3. Split settings

| Setting | Value | Why |
|---|---|---|
| `TEST_FRACTION` | 0.20 | the specification leaves the fraction to this notebook; 20% is the usual choice. It keeps about 4,000 listings for testing, which is enough to report errors per price range and per property type, while leaving about 16,000 for training |
| `RANDOM_STATE` | 42 | any fixed number works. Fixing it makes the split reproducible |
| `groups` | `property_group_id` (v2) | keeps all copies of a property together |

**How `GroupShuffleSplit` works:** it takes the list of unique groups, shuffles it (using `RANDOM_STATE`), and puts
20% of the **groups** into the test set. Every listing then follows its group. So `test_size` is a fraction of
**groups**, not of rows. Because 92% of groups hold a single listing and the largest holds only 10, the share of
**rows** in the test set will still be very close to 20%. We check that below.

Because the groups are shuffled, the order of the rows in the file (which follows price) has no influence on the split.

**Important:** the seed is chosen once and **not** changed afterwards to get a "nicer" split. Trying several seeds and
keeping the one with the best-looking test set would be a way of letting the test set influence our decisions.

In [4]:
TEST_FRACTION = 0.20
RANDOM_STATE = 42

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_STATE)

# split() returns the row POSITIONS (0, 1, 2, ...) of the training and test rows.
# With n_splits=1 there is exactly one pair, and next() takes that first (and only) pair.
train_positions, test_positions = next(splitter.split(listings, groups=listings["property_group_id"]))

listings["split"] = "train"
listings.iloc[test_positions, listings.columns.get_loc("split")] = "test"

print("Training rows:", len(train_positions))
print("Test rows:    ", len(test_positions))

Training rows: 16219
Test rows:     4081


## 4. Validation: no property group on both sides

**What:** the most important check of this notebook. We confirm that:
1. every listing is assigned exactly once (train + test = all rows, and no listing is in both);
2. **no `property_group_id` appears in both train and test**;
3. every multi-listing group sits entirely on one side (checked a second way, group by group).

Each check is an `assert`, so the notebook stops if the split is wrong.

In [5]:
train = listings[listings["split"] == "train"]
test = listings[listings["split"] == "test"]

# 1. Every listing assigned exactly once
assert len(train) + len(test) == len(listings)
assert set(train["property_id"]).isdisjoint(set(test["property_id"]))
assert listings["split"].isin(["train", "test"]).all()

# 2. No group on both sides
train_groups = set(train["property_group_id"])
test_groups = set(test["property_group_id"])
groups_on_both_sides = train_groups & test_groups
print("Property groups in both train and test:", len(groups_on_both_sides))
assert len(groups_on_both_sides) == 0, "A property group appears in both train and test!"

# 3. The same check group by group: every group must have exactly one split value
splits_per_group = listings.groupby("property_group_id")["split"].nunique()
print("Groups with more than one split value: ", (splits_per_group > 1).sum())
assert (splits_per_group == 1).all()

# The multi-listing groups are the ones that matter most for leakage
multi_groups = listings.loc[in_multi_listing_group, "property_group_id"].unique()
print("Multi-listing groups checked:          ", len(multi_groups), "- all on one side")
print()
print("Validation passed: no property group is shared between train and test.")

Property groups in both train and test: 0
Groups with more than one split value:  0
Multi-listing groups checked:           709 - all on one side

Validation passed: no property group is shared between train and test.


## 5. Row and group counts

In [6]:
def count_summary(part):
    part_group_sizes = part["property_group_id"].map(part["property_group_id"].value_counts())
    return {
        "rows": len(part),
        "% of rows": round(len(part) / len(listings) * 100, 2),
        "property groups": part["property_group_id"].nunique(),
        "% of groups": round(part["property_group_id"].nunique() / listings["property_group_id"].nunique() * 100, 2),
        "'single' groups": part["property_group_id"].str.startswith("single ").sum(),
        "groups with 2+ listings": part.loc[part_group_sizes > 1, "property_group_id"].nunique(),
        "rows in groups with 2+ listings": (part_group_sizes > 1).sum(),
    }


counts = pd.DataFrame({"train": count_summary(train), "test": count_summary(test), "all": count_summary(listings)})
counts

,train,test,all
rows,16219.0,4081.0,20300.0
% of rows,79.9,20.1,100.0
property groups,15580.0,3895.0,19475.0
% of groups,80.0,20.0,100.0
'single' groups,1795.0,447.0,2242.0
groups with 2+ listings,550.0,159.0,709.0
rows in groups with 2+ listings,1189.0,345.0,1534.0


## 6. Price distribution in train and test

**What:** compare the asking price in the two sets: percentiles (in ₹ lakh), the mean of log10(price), and the share of
listings per notebook 12 price range (`price_band`).

**Why:** a random split should give two sets with similar prices. If the test set were, for example, much more
expensive, the test score would describe a different population from the one the model was trained on. The price
bands are also the ranges in which errors will later be reported (the sampling doc requires it), so each band needs
enough test listings.

This is a **sanity check, not a rule**: small differences are expected by chance, and we do not re-draw the split.
`price_band` is used here for reporting only. It is derived from the target and is never a feature.

In [7]:
PERCENTILES = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]


def price_summary(part):
    price_lakh = part["price"] / 1e5
    summary = {"listings": len(part), "min": price_lakh.min()}
    for percentile in PERCENTILES:
        summary[f"{percentile:.0%}"] = price_lakh.quantile(percentile)
    summary["max"] = price_lakh.max()
    summary["mean of log10(price)"] = np.log10(part["price"]).mean()
    return summary


price_table = pd.DataFrame({"train": price_summary(train), "test": price_summary(test)})
price_table["test / train"] = price_table["test"] / price_table["train"]
print("Asking price in ₹ lakh (1 lakh = ₹1,00,000; 100 lakh = ₹1 crore)")
price_table.round(3)

Asking price in ₹ lakh (1 lakh = ₹1,00,000; 100 lakh = ₹1 crore)


,train,test,test / train
listings,16219.000,4081.000,0.252
min,1.400,5.500,3.929
1%,25.500,26.700,1.047
5%,45.000,45.000,1.000
25%,95.000,95.000,1.000
50%,150.000,153.000,1.020
75%,228.875,225.000,0.983
95%,550.000,550.000,1.000
99%,1200.000,1200.000,1.000
max,5500.000,4400.000,0.800


In [8]:
RANGE_NAMES = ["0-50 L", "50-80 L", "80 L-1.2 Cr", "1.2-1.5 Cr", "1.5-1.75 Cr",
               "1.75-2 Cr", "2-3 Cr", "3-5 Cr", "5 Cr+"]

band_table = pd.DataFrame({
    "train listings": train["price_band"].value_counts(),
    "test listings": test["price_band"].value_counts(),
    "train %": train["price_band"].value_counts(normalize=True) * 100,
    "test %": test["price_band"].value_counts(normalize=True) * 100,
}).reindex(RANGE_NAMES)
band_table["difference (points)"] = band_table["test %"] - band_table["train %"]
print("Listings per price range (for reporting only)")
band_table.round(2)

Listings per price range (for reporting only)


,train listings,test listings,train %,test %,difference (points)
price_band,,,,,
0-50 L,1260,311,7.77,7.62,-0.15
50-80 L,2291,602,14.13,14.75,0.63
80 L-1.2 Cr,2402,529,14.81,12.96,-1.85
1.2-1.5 Cr,2287,585,14.10,14.33,0.23
1.5-1.75 Cr,1687,458,10.40,11.22,0.82
1.75-2 Cr,1616,436,9.96,10.68,0.72
2-3 Cr,2162,534,13.33,13.09,-0.25
3-5 Cr,1550,398,9.56,9.75,0.20
5 Cr+,964,228,5.94,5.59,-0.36


## 7. Property-type distribution in train and test

**What:** compare the share of each property type in the two sets.

**Why:** area, floor and price mean different things for flats, houses and villas (notebooks 08 and 13), and errors
will later be reported per property type. Each type should appear in similar proportions on both sides. Penthouses
are rare (48 listings in the whole data), so their test count will be small and their results noisy.

In [9]:
PROPERTY_TYPES = ["Apartment", "Residential House", "Villa", "Builder Floor Apartment", "Penthouse"]

type_table = pd.DataFrame({
    "train listings": train["property_type"].value_counts(),
    "test listings": test["property_type"].value_counts(),
    "train %": train["property_type"].value_counts(normalize=True) * 100,
    "test %": test["property_type"].value_counts(normalize=True) * 100,
}).reindex(PROPERTY_TYPES)
type_table["difference (points)"] = type_table["test %"] - type_table["train %"]
type_table.round(2)

,train listings,test listings,train %,test %,difference (points)
property_type,,,,,
Apartment,9218,2328,56.83,57.04,0.21
Residential House,4353,1093,26.84,26.78,-0.06
Villa,2116,503,13.05,12.33,-0.72
Builder Floor Apartment,498,148,3.07,3.63,0.56
Penthouse,34,9,0.21,0.22,0.01


**Median asking price per property type.** This checks that each type is priced similarly on both sides, not just
present in similar numbers.

In [10]:
median_by_type = pd.DataFrame({
    "train median (₹ lakh)": train.groupby("property_type")["price"].median() / 1e5,
    "test median (₹ lakh)": test.groupby("property_type")["price"].median() / 1e5,
}).reindex(PROPERTY_TYPES)
median_by_type["test / train"] = median_by_type["test median (₹ lakh)"] / median_by_type["train median (₹ lakh)"]
median_by_type.round(2)

,train median (₹ lakh),test median (₹ lakh),test / train
property_type,,,
Apartment,128.84,130.00,1.01
Residential House,175.00,168.75,0.96
Villa,270.10,282.59,1.05
Builder Floor Apartment,73.50,79.50,1.08
Penthouse,120.00,105.00,0.88


## 8. Save the split assignment

**What:** save one row per listing with `property_id`, `property_group_id` and `split` (`train` or `test`).

**Why:** later notebooks join this file to `cleaned_listings.csv` on `property_id`, so every experiment uses **exactly**
the same test set. We save the assignment, not two copies of the data, so there is only one version of each listing.
`property_group_id` is included so later cross-validation can use `GroupKFold` with the same groups inside the training set.

We then load the file again and check it matches what we saved, and check that the cleaned file was not modified.

In [11]:
split_assignment = listings[["property_id", "property_group_id", "split"]]
split_assignment.to_csv(SPLIT_CSV, index=False)
print("Saved:", SPLIT_CSV.relative_to(PROJECT_ROOT), split_assignment.shape)
print(split_assignment["split"].value_counts().to_string())

reloaded = pd.read_csv(SPLIT_CSV)
assert len(reloaded) == len(listings)
assert (reloaded["property_id"].values == listings["property_id"].values).all()
assert (reloaded["split"].values == listings["split"].values).all()
assert reloaded["property_id"].is_unique
reloaded_train_groups = set(reloaded.loc[reloaded["split"] == "train", "property_group_id"])
reloaded_test_groups = set(reloaded.loc[reloaded["split"] == "test", "property_group_id"])
assert reloaded_train_groups.isdisjoint(reloaded_test_groups)
print("Reloaded file matches, and still has no group on both sides.")

Saved: data\processed\train_test_split.csv (20300, 3)
split
train    16219
test      4081
Reloaded file matches, and still has no group on both sides.


In [12]:
cleaned_md5_at_end = file_md5(CLEANED_CSV)
print("MD5 of cleaned_listings.csv at start:", cleaned_md5_at_start)
print("MD5 of cleaned_listings.csv at end:  ", cleaned_md5_at_end)
assert cleaned_md5_at_end == cleaned_md5_at_start, "cleaned_listings.csv changed!"
print("cleaned_listings.csv is unchanged.")

MD5 of cleaned_listings.csv at start: 3a81a385a9aca28d5b70322e1e143c08
MD5 of cleaned_listings.csv at end:   3a81a385a9aca28d5b70322e1e143c08
cleaned_listings.csv is unchanged.


## 9. Summary and how to use the split later

**Done here:**
- A group-aware split with `GroupShuffleSplit` (`test_size=0.20`, `random_state=42`, `groups=property_group_id`).
- Checked: every listing is assigned once, and **no property group appears in both train and test**.
- Compared train and test on row and group counts, price distribution, price ranges and property types.
- Saved the assignment to `data/processed/train_test_split.csv`.

**How later notebooks should use it:**

```python
listings = pd.read_csv("data/processed/cleaned_listings.csv")
split = pd.read_csv("data/processed/train_test_split.csv")
listings = listings.merge(split[["property_id", "split"]], on="property_id", validate="one_to_one")
train = listings[listings["split"] == "train"]
test = listings[listings["split"] == "test"]
```

**Rules from here on** (spec sections 13 and 14):
- Fit every learned step (imputation, encoding, rare-locality grouping, scaling, any locality price encoding) on
  `train` only, preferably inside a scikit-learn `Pipeline`.
- For model selection, use cross-validation **inside the training set** with `GroupKFold` and
  `groups = property_group_id`, so copies of a property never sit in two folds.
- Do not look at test-set errors until the final model is chosen. The test set is used **once**.
- Report errors overall, per `price_band` and per property type.
- Only columns marked "possible feature" in `cleaned_listings_column_roles.csv` may become model inputs.